# Final plots — NewSouthWales showcase

The "epic summary" figures for this case: the ones meant to represent it in the
**GUI results portal**, not working diagnostics. Last step in the pipeline, after
validation (09–11).

**Story arc**
1. Domain — goals on the open boundary, WHACS hindcast links, buoy
2. Offshore forcing — bulk + partition Hs time series at the buoy's goal
3. LHS training — sampled SnapWave cases per goal
4. WHACS hindcast — observed bulk forcing at all goals (dynamic mode)
5. Buoy context — validation site on the domain map
6. Hero spectrum — buoy vs bulk vs partition reconstruct at storm peak
7. Pipeline schematic — workflow overview
8. Input / output — partition arrows at goals + reconstructed spectra at buoy
9. Holdout validation summary — R² across all 26 goals
10. Buoy network overview — all 35 observation sites
11. Reconstruction spectra network — spectra bubbles along the 5 m isobath +
    forcing arrows, for one moment

**Outputs:** `outputs/gui_figures/*.png` — kept at this path since that's what
the case contract and the GUI actually read (`CONTRACT.md`, `GUI/README.md`); only
the notebook that produces them moved.

**Prerequisites:** goals GPKG, WHACS zarr, SnapWave LHS cases, fitted models,
reconstruction caches for the chosen `period` (notebooks 05–07, 10–11);
`outputs/goal_*/validation_metrics.csv` for Fig 9 (notebook 06).

In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from utils.config import (
    PATHS,
    SIMULATIONS,
    goals_for_snapwave,
    load_goals,
    open_offshore_waves,
    resolve,
)
from utils.evaluation import validation as val
from utils.geometry.output_sites import compute_output_sites
from utils.model import metamodel as mm
from utils.observation import onshore_sites as obs
from utils.simulation.snapwave_wrappers import (
    build_enclosure_polygon,
    read_polygon_from_shp_file,
)
from utils.viz import case_plots as cp
from utils.viz.spectra_plots import (
    plot_spectrum_comparison,
    render_polar_spectrum,
    render_spectrum_bubble,
)

## Configuration & helpers


In [2]:
# --- slide / export settings ---
EXPORT_DIR = resolve("outputs/gui_figures")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DPI = 200
FIGSIZE_WIDE = (13.33, 7.5)  # 16:9
FIGSIZE_TALL = (10.0, 11.0)

# --- case study ---
# period/buoy match the combination already confirmed to have working
# reconstruction spectra caches in 11_Validate_Spectra.ipynb.
period = "2011"
storm_days = 5
forcing_moment = (
    None  # None → storm peak at buoy goal  # window around peak Hs for partition plot
)
offshore_name = "WHACS"
mda_n = 450
pca_variance = 0.99
wl = 0.0

buoy_instrument = "ByronBay_Buoy"  # instrument_id from the observations GeoJSON
buoy_site = obs.snapwave_site_id(buoy_instrument)
buoy_nc = "inputs/Observations/processed/spectra/ByronBay_Buoy.nc"

recon_kw = dict(
    n=mda_n,
    offshore_name=offshore_name,
    goals=None,
    wl=wl,
    site_ids=[buoy_site],
    pca_variance=pca_variance,
)

GOALS_GPKG = resolve(SIMULATIONS["goals_file"]).with_suffix(".gpkg")
goals = load_goals()
goals_config = goals_for_snapwave(goals)
targets = gpd.read_file(GOALS_GPKG, layer="targets")
boundary_snaps_gdf = gpd.read_file(GOALS_GPKG, layer="boundary_snaps")
boundary_snaps = {
    str(int(row.goal_id)): {
        "longitude": float(row.geometry.x),
        "latitude": float(row.geometry.y),
    }
    for _, row in boundary_snaps_gdf.iterrows()
}

aoi_gdf, aoi_polygon = read_polygon_from_shp_file(resolve(PATHS["aoi_file"]))
enclosure_polygon = build_enclosure_polygon(
    aoi_polygon,
    float(SIMULATIONS.get("enclosure_buffer_m", 10_000)),
    n_vertices=4,
)
output_sites_df = compute_output_sites(aoi_polygon=aoi_polygon)

goal_map = val.closest_goals_table(
    [buoy_site], offshore_names=[offshore_name], goals=goals
)
buoy_goal = int(goal_map.goal.iloc[0])
print(f"Export dir : {EXPORT_DIR}")
print(f"Period     : {period}")
print(f"Buoy site  : {buoy_site}  →  closest goal G{buoy_goal}")


def apply_slide_style() -> None:
    plt.rcParams.update(
        {
            "font.size": 13,
            "axes.titlesize": 15,
            "axes.labelsize": 13,
            "legend.fontsize": 11,
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
        }
    )


def save_fig(fig, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"saved → {path}")
    return path


def build_lhs_cases_df() -> pd.DataFrame:
    """Forcing table from SnapWave LHS NetCDFs (one file per goal)."""
    chunks: list[pd.DataFrame] = []
    for gid in sorted(goals, key=int):
        with xr.open_dataset(mm.cases_path(gid)) as ds:
            df = (
                ds[["tp_forcing", "dir_forcing", "spr_forcing", "wl_forcing"]]
                .to_dataframe()
                .reset_index()
            )
        df["goal"] = int(gid)
        df["tp"] = df["tp_forcing"]
        df["dir"] = df["dir_forcing"]
        df["spr"] = df["spr_forcing"]
        df["wl"] = df["wl_forcing"]
        chunks.append(df[["goal", "tp", "dir", "spr", "wl"]])
    return pd.concat(chunks, ignore_index=True)


def pick_storm_peak_time(goal_id: int) -> pd.Timestamp:
    off = open_offshore_waves(offshore_name).sel(time=period, goal=goal_id)
    return pd.Timestamp(off["hs"].idxmax("time").values)


def storm_time_slice(peak: pd.Timestamp, days: int) -> slice:
    half = pd.Timedelta(days=days / 2)
    return slice(peak - half, peak + half)


apply_slide_style()

PARTITION_STYLE = {
    0: ("Wind sea (p0)", "#e41a1c"),
    1: ("Swell 1 (p1)", "#377eb8"),
    2: ("Swell 2 (p2)", "#4daf4a"),
    3: ("Swell 3 (p3)", "#984ea3"),
}


def resolve_forcing_moment(goal_id: int | None = None) -> pd.Timestamp:
    if forcing_moment is not None:
        return pd.Timestamp(forcing_moment)
    gid = buoy_goal if goal_id is None else goal_id
    return pick_storm_peak_time(gid)


def offshore_partitions_at(time: pd.Timestamp) -> xr.Dataset:
    return open_offshore_waves(offshore_name).sel(time=time, method="nearest")


def _partition_arrowprops(
    lon: float,
    lat: float,
    dir_deg: float,
    *,
    scale_deg: float,
    color: str,
    lw: float = 2.4,
    alpha: float = 0.92,
):
    """Arrow style with larger heads for course maps."""
    theta = np.deg2rad(float(dir_deg))
    dlon = np.sin(theta) * scale_deg
    dlat = np.cos(theta) * scale_deg
    return (
        dict(
            arrowstyle="-|>",
            color=color,
            lw=lw,
            alpha=alpha,
            shrinkA=0,
            shrinkB=2,
            mutation_scale=16,
        ),
        (lon + dlon, lat + dlat),
        (lon, lat),
    )


def course_map_extent(
    *, pad_lon: float = 0.15, pad_lat: float = 0.12
) -> tuple[float, float, float, float]:
    """Lon/lat limits that include AoI, all goals, WHACS nodes, and buoy."""
    lons: list[float] = []
    lats: list[float] = []
    for g in goals.values():
        lons.append(g["target"]["longitude"])
        lats.append(g["target"]["latitude"])
        hc = g["hindcast"]["whacs"]
        lons.append(hc["longitude"])
        lats.append(hc["latitude"])
    for snap in boundary_snaps.values():
        lons.append(snap["longitude"])
        lats.append(snap["latitude"])
    buoy = obs.get_coordinates()[buoy_instrument]
    lons.append(buoy["lon"])
    lats.append(buoy["lat"])
    bb = aoi_gdf.total_bounds
    lons.extend([float(bb[0]), float(bb[2])])
    lats.extend([float(bb[1]), float(bb[3])])
    return (
        min(lons) - pad_lon,
        max(lons) + pad_lon,
        min(lats) - pad_lat,
        max(lats) + pad_lat,
    )


def partition_legend_handles() -> list:
    """Legend entries with arrow glyphs for partition colours."""
    handles = []
    for part, (label, color) in sorted(PARTITION_STYLE.items()):
        handles.append(
            plt.Line2D(
                [0],
                [0],
                color=color,
                linewidth=3.2,
                marker=">",
                markersize=11,
                markeredgewidth=0,
                label=f"{label}  (dir = wave-from, len ∝ Hs)",
            )
        )
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="black",
            markersize=7,
            linestyle="None",
            label="goal boundary snap",
        )
    )
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=cp.TYPE_COLORS["validation"],
            markersize=10,
            linestyle="None",
            label="buoy (spectra above)",
        )
    )
    return handles


def draw_goal_partition_arrows(
    ax,
    time: pd.Timestamp,
    *,
    arrow_base_deg: float = 0.30,
    hs_floor: float = 0.04,
    fan_offset_deg: float = 0.028,
) -> None:
    """Four partition arrows per goal (wave-from direction, length ∝ phs)."""
    off = offshore_partitions_at(time)
    goal_ids = sorted(int(g) for g in off.goal.values)
    part_vars = [f"phs{i}" for i in range(4)]
    hs_max = float(np.nanmax(off[part_vars].to_array("part").values))
    hs_max = max(hs_max, hs_floor)

    for gid in goal_ids:
        key = str(gid)
        snap = boundary_snaps[key]
        lon, lat = snap["longitude"], snap["latitude"]
        ax.plot(
            lon,
            lat,
            "o",
            color="white",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=1.0,
            zorder=6,
        )
        ax.annotate(
            f"G{gid}",
            (lon, lat),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            color="#111111",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75),
            zorder=7,
        )

        for part in range(4):
            phs = float(off[f"phs{part}"].sel(goal=gid).values)
            pdir = float(off[f"pdir{part}"].sel(goal=gid).values)
            if not np.isfinite(phs) or phs < hs_floor or not np.isfinite(pdir):
                continue
            color = PARTITION_STYLE[part][1]
            scale = arrow_base_deg * (0.45 + 0.55 * phs / hs_max)
            theta = np.deg2rad(pdir)
            perp = theta + np.pi / 2
            shift = (part - 1.5) * fan_offset_deg
            lon0 = lon + shift * np.sin(perp)
            lat0 = lat + shift * np.cos(perp)
            props, xytext, xy = _partition_arrowprops(
                lon0,
                lat0,
                pdir,
                scale_deg=scale,
                color=color,
            )
            ax.annotate("", xy=xy, xytext=xytext, arrowprops=props, zorder=5)

Export dir : /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures
Period     : 2011
Buoy site  : buoy_ByronBay_Buoy  →  closest goal G4


## Fig 1 — Domain map

Goals on the AoI boundary (propagation wedges), dashed links to **WHACS** hindcast
nodes, and the validation buoy inside the domain.


In [3]:
apply_slide_style()
goal_ids = sorted(int(k) for k in goals)
colors = cp._goal_colors(goal_ids)
buoy_meta = obs.get_coordinates()[buoy_instrument]

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
cp.plot_base_map(ax=ax, title="Offshore goals & buoy", show_topobathy=True)
cp.plot_sectors(targets, goals, boundary_snaps, ax, radius_deg=0.28)

for gid in goal_ids:
    key = str(gid)
    snap = boundary_snaps[key]
    hc = goals[key]["hindcast"]["whacs"]
    color = colors[gid]
    ax.plot(
        [snap["longitude"], hc["longitude"]],
        [snap["latitude"], hc["latitude"]],
        color=color,
        linewidth=1.2,
        alpha=0.75,
        zorder=5,
    )
    ax.plot(
        hc["longitude"],
        hc["latitude"],
        marker="o",
        color=color,
        markersize=5,
        markeredgecolor="white",
        markeredgewidth=0.4,
        zorder=6,
    )

ax.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    marker="s",
    color=cp.TYPE_COLORS["validation"],
    markersize=12,
    markeredgecolor="white",
    markeredgewidth=0.8,
    zorder=8,
    label="validation buoy",
)
ax.annotate(
    "buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(6, 6),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    weight="bold",
)

handles = [
    mpatches.Patch(facecolor=colors[g], alpha=0.5, label=f"Goal {g}")
    for g in goal_ids[:4]
]
handles.append(
    plt.Line2D([0], [0], color="gray", linestyle="--", label="target → boundary")
)
handles.append(
    plt.Line2D([0], [0], color="gray", linewidth=1.2, label="boundary → WHACS")
)
handles.append(
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor=cp.TYPE_COLORS["validation"],
        markersize=10,
        label="buoy",
    )
)
ax.legend(handles=handles, loc="lower left", fontsize=9, framealpha=0.92)
x0, x1, y0, y1 = course_map_extent()
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)

save_fig(fig, "fig01_domain")

saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig01_domain.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig01_domain.png')

## Fig 2 — Offshore partitions (time series)

WHACS **bulk Hs** and partition components ``phs0``–``phs3`` at the goal closest
to the buoy, centred on the storm peak in ``period``.


In [4]:
apply_slide_style()
peak_time = pick_storm_peak_time(buoy_goal)
t_slice = storm_time_slice(peak_time, storm_days)

off = open_offshore_waves(offshore_name).sel(time=t_slice, goal=buoy_goal).load()
times = pd.to_datetime(off.time.values)

partition_style = {
    "phs0": ("Wind sea (p0)", "#e41a1c"),
    "phs1": ("Swell 1 (p1)", "#377eb8"),
    "phs2": ("Swell 2 (p2)", "#4daf4a"),
    "phs3": ("Swell 3 (p3)", "#984ea3"),
}

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
for var, (label, color) in partition_style.items():
    if var in off:
        ax.plot(times, off[var].values, label=label, color=color, linewidth=1.6)

ax.plot(
    times,
    off["hs"].values,
    label="Bulk Hs (p−1)",
    color="black",
    linewidth=2.2,
    linestyle="--",
)
ax.axvline(peak_time, color="gray", linestyle=":", linewidth=1.2, label="storm peak")
ax.set_title(
    f"WHACS offshore Hs — goal G{buoy_goal} — {period}\n"
    f"peak @ {peak_time:%Y-%m-%d %H:%M} UTC"
)
ax.set_xlabel("Time")
ax.set_ylabel("Hs (m)")
ax.legend(loc="upper right", ncol=2)
ax.grid(True, alpha=0.25)

save_fig(fig, "fig02_offshore_partitions_ts")

saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig02_offshore_partitions_ts.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig02_offshore_partitions_ts.png')

## Fig 3 — Metamodel training (LHS cases)

Sampled wave directions per goal from the **500 LHS SnapWave cases** (notebook 05).
Reuses the same diagnostic layout as case generation.


In [5]:
apply_slide_style()
lhs_cases = build_lhs_cases_df()
print(
    f"LHS cases loaded: {len(lhs_cases):,} rows across {lhs_cases['goal'].nunique()} goals"
)

fig = cp.plot_metamodel_configuration(
    aoi_gdf=aoi_gdf,
    enclosure_polygon=enclosure_polygon,
    goals_config=goals_config,
    cases_df=lhs_cases,
    output_sites_df=output_sites_df,
    bounds=SIMULATIONS.get("bounds"),
    num_samples_per_goal=SIMULATIONS.get("num_samples_per_goal"),
    mesh_file=PATHS.get("mesh_file"),
    enclosure_buffer_m=float(SIMULATIONS.get("enclosure_buffer_m", 10_000)),
    sector_radius_deg=0.30,
    arrow_scale_deg=0.12,
    max_arrows_per_goal=18,
    show_topobathy=True,
    title="Metamodel training — LHS cases per goal",
)
save_fig(fig, "fig03_metamodel_lhs")

LHS cases loaded: 12,974 rows across 26 goals
saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig03_metamodel_lhs.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig03_metamodel_lhs.png')

## Fig 4 — WHACS hindcast forcing (dynamic mode)

Observed bulk forcing from WHACS at every goal for ``period`` — the input that
drives **dynamic** SnapWave runs and **reconstruction** hindcasts.


In [6]:
apply_slide_style()
offshore = open_offshore_waves(offshore_name).sel(time=period)

fig = cp.plot_dynamic_configuration(
    aoi_gdf=aoi_gdf,
    enclosure_polygon=enclosure_polygon,
    goals_config=goals_config,
    offshore=offshore,
    output_sites_df=output_sites_df,
    offshore_name=offshore_name,
    mesh_file=PATHS.get("mesh_file"),
    enclosure_buffer_m=float(SIMULATIONS.get("enclosure_buffer_m", 10_000)),
    sector_radius_deg=0.30,
    arrow_scale_deg=0.12,
    max_arrows_per_goal=18,
    max_dir_marks=150,
    show_topobathy=True,
    title=f"WHACS hindcast forcing — {period}",
)
save_fig(fig, "fig04_whacs_hindcast_forcing")

saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig04_whacs_hindcast_forcing.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig04_whacs_hindcast_forcing.png')

## Fig 5 — Buoy on domain (validation context)

Zoomed map highlighting the buoy, its closest goal, and the WHACS node used for
reconstruction at that goal.


In [7]:
apply_slide_style()
buoy_meta = obs.get_coordinates()[buoy_instrument]
row = goal_map.iloc[0]
gkey = str(buoy_goal)
snap = boundary_snaps[gkey]
hc = goals[gkey]["hindcast"]["whacs"]
tgt = goals[gkey]["target"]

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
cp.plot_base_map(ax=ax, title="Buoy validation site", show_topobathy=True)

left, right = goals[gkey]["angles"]
theta1, theta2 = cp.nautical_sector_to_wedge_angles(left, right)
ax.add_patch(
    mpatches.Wedge(
        (snap["longitude"], snap["latitude"]),
        0.18,
        theta1,
        theta2,
        facecolor="#1f78b4",
        alpha=0.25,
        edgecolor="#1f78b4",
        linewidth=1.0,
    )
)
ax.plot(
    [tgt["longitude"], snap["longitude"]],
    [tgt["latitude"], snap["latitude"]],
    "k--",
    alpha=0.5,
    linewidth=0.9,
)
ax.plot(
    [snap["longitude"], hc["longitude"]],
    [snap["latitude"], hc["latitude"]],
    color="#1f78b4",
    linewidth=1.8,
)
ax.plot(
    hc["longitude"],
    hc["latitude"],
    "o",
    color="#1f78b4",
    markersize=9,
    markeredgecolor="white",
)
ax.plot(snap["longitude"], snap["latitude"], "kx", markersize=7)
ax.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    "s",
    color=cp.TYPE_COLORS["validation"],
    markersize=14,
    markeredgecolor="white",
    markeredgewidth=0.8,
    zorder=8,
)
ax.annotate(
    f"G{buoy_goal} WHACS",
    (hc["longitude"], hc["latitude"]),
    xytext=(8, -10),
    textcoords="offset points",
    fontsize=10,
)
ax.annotate(
    "buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(8, 6),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    weight="bold",
)

pad = 0.35
ax.set_xlim(
    min(buoy_meta["lon"], hc["longitude"]) - pad,
    max(buoy_meta["lon"], hc["longitude"]) + pad,
)
ax.set_ylim(
    min(buoy_meta["lat"], hc["latitude"]) - pad,
    max(buoy_meta["lat"], hc["latitude"]) + pad,
)
ax.set_aspect("equal", adjustable="box")
ax.text(
    0.02,
    0.02,
    f"closest goal: G{buoy_goal}  ({row.distance_m / 1000:.1f} km to WHACS node)",
    transform=ax.transAxes,
    fontsize=10,
    va="bottom",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
)

save_fig(fig, "fig05_buoy_context_map")

saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig05_buoy_context_map.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig05_buoy_context_map.png')

## Fig 6 — Hero spectrum (storm peak)

Polar spectra at the buoy for the **storm-peak hour**: observed buoy, bulk
reconstruct (partition −1), and partition-sum reconstruct (0–3).

Requires v5 reconstruction spectra caches for ``period``.


In [8]:
apply_slide_style()
peak_time = pick_storm_peak_time(buoy_goal)
print(f"Hero spectrum time: {peak_time}")

buoy_raw = xr.open_dataset(resolve(buoy_nc))
spec_var = "Efth" if "Efth" in buoy_raw else "efth"
buoy_efth = buoy_raw[spec_var].sel(time=peak_time, method="nearest")

bulk_spec = mm.load_reconstruction_spectra(
    period, spectral=False, load=False, **recon_kw
)
part_spec = mm.load_reconstruction_spectra(
    period, spectral=True, load=False, **recon_kw
)
for label, ds in [("bulk", bulk_spec), ("partition", part_spec)]:
    if ds is None:
        raise FileNotFoundError(
            f"Missing {label} spectra cache for {period!r}. Run "
            f"mm.reconstruct(..., spectral={label == 'partition'}, recompute=True)."
        )

bulk_efth = bulk_spec["efth"].sel(sites=buoy_site, time=peak_time, method="nearest")
part_efth = part_spec["efth"].sel(sites=buoy_site, time=peak_time, method="nearest")

panels = [
    (f"Buoy observed — {peak_time:%Y-%m-%d %H:%M}", buoy_efth),
    ("Reconstruction bulk (partition −1)", bulk_efth),
    ("Reconstruction spectral (partitions 0–3 sum)", part_efth),
]
combined = plot_spectrum_comparison(
    panels,
    cbar_kwargs={"label": "Normalised energy density", "shrink": 0.78, "pad": 0.06},
)

out = EXPORT_DIR / "fig06_hero_spectrum_storm_peak.png"
combined.save(out)
print(f"saved → {out}")

Hero spectrum time: 2011-05-25 19:00:00


FileNotFoundError: Missing bulk spectra cache for '2011'. Run mm.reconstruct(..., spectral=False, recompute=True).

## Fig 7 — Metamodel pipeline (schematic)

Simple flow diagram for slides — no model data required.


In [9]:
apply_slide_style()
steps = [
    "WHACS offshore\n(goals × partitions)",
    "MDA + LHS\n500 cases / goal",
    "SnapWave\nnearshore cases",
    "PCA + GPR\nmetamodel",
    "mm.reconstruct()\nhindcast",
    "Buoy / site\nvalidation",
]
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
ax.set_xlim(0, len(steps))
ax.set_ylim(0, 1)
ax.axis("off")

for i, text in enumerate(steps):
    ax.text(
        i + 0.5,
        0.55,
        text,
        ha="center",
        va="center",
        fontsize=12,
        bbox=dict(
            boxstyle="round,pad=0.45",
            facecolor="#e8f4fc",
            edgecolor="#1f78b4",
            linewidth=1.2,
        ),
    )
    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(i + 1.08, 0.55),
            xytext=(i + 0.92, 0.55),
            arrowprops=dict(arrowstyle="-|>", color="#444444", lw=1.8),
        )

ax.set_title("HyGloW — hybrid downscaling workflow", fontsize=16, pad=12)
save_fig(fig, "fig07_pipeline_schematic")

saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig07_pipeline_schematic.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig07_pipeline_schematic.png')

## Fig 8 — Input / output at one moment

**Top (output):** reconstructed spectra at the buoy for the same UTC hour.

**Bottom (input):** at each goal boundary snap, four coloured arrows show WHACS
partition forcing (``pdir`` + ``phs``) — wind sea + three swells.

Set ``forcing_moment`` in the config cell to pin a specific time; default is the
storm peak at the buoy's closest goal.


In [10]:
apply_slide_style()
moment = resolve_forcing_moment()
print(f"Forcing moment: {moment}")

# --- output spectra (buoy) ---
buoy_raw = xr.open_dataset(resolve(buoy_nc))
spec_var = "Efth" if "Efth" in buoy_raw else "efth"
buoy_efth = buoy_raw[spec_var].sel(time=moment, method="nearest")

bulk_spec = mm.load_reconstruction_spectra(
    period, spectral=False, load=False, **recon_kw
)
part_spec = mm.load_reconstruction_spectra(
    period, spectral=True, load=False, **recon_kw
)
for label, ds in [("bulk", bulk_spec), ("partition", part_spec)]:
    if ds is None:
        raise FileNotFoundError(
            f"Missing {label} spectra cache for {period!r}. Run "
            f"mm.reconstruct(..., spectral={label == 'partition'}, recompute=True)."
        )

bulk_efth = bulk_spec["efth"].sel(sites=buoy_site, time=moment, method="nearest")
part_efth = part_spec["efth"].sel(sites=buoy_site, time=moment, method="nearest")

spec_panels = [
    (buoy_efth, "Buoy observed"),
    (bulk_efth, "Reconstruct bulk (p−1)"),
    (part_efth, "Reconstruct partitions 0–3"),
]

# constrained_layout avoids the empty gap GridSpec + height_ratios otherwise
# leaves between the suptitle and the top row (and between the two rows).
fig = plt.figure(figsize=(16.5, 12.0), constrained_layout=True)
gs = gridspec.GridSpec(
    2,
    3,
    figure=fig,
    height_ratios=[1.2, 2.5],
    hspace=0.05,
    wspace=0.05,
)

for col, (da, title) in enumerate(spec_panels):
    ax_spec = fig.add_subplot(gs[0, col])
    img = render_polar_spectrum(da, title, figsize=(6.0, 5.6), cmap="magma")
    ax_spec.imshow(np.asarray(img))
    ax_spec.axis("off")

ax_map = fig.add_subplot(gs[1, :])
cp.plot_base_map(ax=ax_map, title=None, show_topobathy=True)
draw_goal_partition_arrows(ax_map, moment)

buoy_meta = obs.get_coordinates()[buoy_instrument]
ax_map.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    marker="s",
    color=cp.TYPE_COLORS["validation"],
    markersize=14,
    markeredgecolor="white",
    markeredgewidth=0.9,
    zorder=9,
)
ax_map.annotate(
    "validation buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(8, 8),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    fontweight="bold",
    zorder=10,
    bbox=dict(
        boxstyle="round,pad=0.2", fc="white", ec=cp.TYPE_COLORS["validation"], alpha=0.9
    ),
)

x0, x1, y0, y1 = course_map_extent()
ax_map.set_xlim(x0, x1)
ax_map.set_ylim(y0, y1)
ax_map.set_aspect("equal", adjustable="box")

# NSW's coastline (and goal chain) runs diagonally end to end, so no corner
# *inside* the map axes is reliably clear of goal labels or the buoy marker.
# A figure-level legend placed outside the axes (constrained_layout reserves
# the margin for it automatically) never overlaps map content regardless of
# how the goal chain is angled.
leg = fig.legend(
    handles=partition_legend_handles(),
    loc="outside right upper",
    fontsize=9.5,
    framealpha=0.96,
    edgecolor="#bbbbbb",
    title="WHACS partition forcing (input)",
    title_fontsize=10.5,
    ncol=1,
    borderpad=0.8,
)
leg.get_title().set_fontweight("bold")

fig.suptitle(
    f"INPUT → OUTPUT  |  {moment:%Y-%m-%d %H:%M} UTC  |  spectra at {buoy_site}",
    fontsize=16,
    fontweight="bold",
)
ax_map.set_title(
    "INPUT — partition arrows at each offshore goal  (direction = wave-from, length ∝ Hs)",
    fontsize=13,
    pad=10,
    color="#1a1a2e",
)

save_fig(fig, "fig08_input_output_partitions")

Forcing moment: 2011-05-25 19:00:00


FileNotFoundError: Missing bulk spectra cache for '2011'. Run mm.reconstruct(..., spectral=False, recompute=True).

## Fig 9 — Holdout validation summary (all goals)

Metamodel holdout **R²** for Hs and Dir at the 5 m isobath, one bar pair per goal —
the evidence behind the README's "holdout R² ≈ 0.95–0.98" headline, shown goal by
goal rather than as a single averaged number. Tp is left out of the plot: its R² is
> 0.9999 everywhere because SnapWave (and this metamodel) preserve peak period by
construction, so it doesn't carry downscaling-skill information the way Hs/Dir do.

**Prerequisites:** `outputs/goal_*/validation_metrics.csv` (notebook 06).

In [11]:
apply_slide_style()

metric_files = sorted((EXPORT_DIR.parent).glob("goal_*/validation_metrics.csv"))
if not metric_files:
    raise FileNotFoundError(
        "No outputs/goal_*/validation_metrics.csv found — run notebook 06 "
        "(metamodel fit + holdout validation) for every goal first."
    )

metrics = pd.concat((pd.read_csv(f) for f in metric_files), ignore_index=True)
holdout = metrics[
    (metrics["group"] == "5m") & (metrics["variable"].isin(["hs", "dir"]))
]
r2 = holdout.pivot_table(index="goal", columns="variable", values="r2").sort_index()
print(
    f"Goals covered: {len(r2)}  |  Hs R² range: {r2['hs'].min():.3f}-{r2['hs'].max():.3f}"
    f"  |  Dir R² range: {r2['dir'].min():.3f}-{r2['dir'].max():.3f}"
)

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
x = np.arange(len(r2))
width = 0.38
ax.bar(x - width / 2, r2["hs"], width, label="Hs", color=cp.TYPE_COLORS["waves"])
ax.bar(x + width / 2, r2["dir"], width, label="Dir", color=cp.TYPE_COLORS["spectrum"])
ax.axhline(0.9, color="gray", linestyle=":", linewidth=1.2, label="R² = 0.90")
ax.set_xticks(x)
ax.set_xticklabels([f"G{g}" for g in r2.index], fontsize=9)
ax.set_ylim(min(0.6, r2.min().min() - 0.05), 1.02)
ax.set_ylabel("Holdout R²")
ax.set_xlabel("Goal (north → south, G1 = Byron Bay end, G26 = Eden end)")
ax.set_title(
    "Metamodel holdout validation — R² at every goal (5 m isobath)\n"
    "Tp not shown: R² > 0.9999 everywhere — preserved by construction, not a downscaling skill measure"
)
ax.legend(loc="lower right", ncol=3)
ax.grid(True, axis="y", alpha=0.25)

save_fig(fig, "fig09_holdout_r2_summary")

Goals covered: 26  |  Hs R² range: 0.793-0.972  |  Dir R² range: 0.966-0.990
saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig09_holdout_r2_summary.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig09_holdout_r2_summary.png')

## Fig 10 — Buoy observation network

All 35 sites in the observations catalogue (notebook 03 / `00_Buoys_Inspection.ipynb`):
**7 offshore MHL Waverider buoys** (long-running reference records) and **28 nearshore
DCCEEW Spotter deployments** (shorter, closer to shore). The single buoy used for
spectral validation elsewhere in this notebook is called out separately.

In [12]:
apply_slide_style()

buoy_sites = obs.get_coordinates()
offshore_sites = {
    k: v
    for k, v in buoy_sites.items()
    if (v.get("institution") or "").strip().upper() == "MHL"
}
nearshore_sites = {k: v for k, v in buoy_sites.items() if k not in offshore_sites}
print(f"Offshore MHL buoys      : {len(offshore_sites)}")
print(f"Nearshore Spotter buoys : {len(nearshore_sites)}")

buoy_meta = obs.get_coordinates()[buoy_instrument]

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE, constrained_layout=True)
cp.plot_base_map(ax=ax, title="Buoy observation network", show_topobathy=True)

for meta in nearshore_sites.values():
    ax.plot(
        meta["lon"],
        meta["lat"],
        marker="o",
        color=cp.TYPE_COLORS["calibration"],
        markersize=7,
        markeredgecolor="white",
        markeredgewidth=0.6,
        zorder=6,
    )
for meta in offshore_sites.values():
    ax.plot(
        meta["lon"],
        meta["lat"],
        marker="^",
        color=cp.TYPE_COLORS["waves"],
        markersize=10,
        markeredgecolor="white",
        markeredgewidth=0.7,
        zorder=7,
    )

ax.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    marker="s",
    color=cp.TYPE_COLORS["validation"],
    markersize=13,
    markeredgecolor="white",
    markeredgewidth=0.8,
    zorder=9,
)
ax.annotate(
    f"{buoy_instrument}\n(spectral validation)",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(8, 8),
    textcoords="offset points",
    fontsize=10,
    color=cp.TYPE_COLORS["validation"],
    fontweight="bold",
    zorder=10,
    bbox=dict(
        boxstyle="round,pad=0.2", fc="white", ec=cp.TYPE_COLORS["validation"], alpha=0.9
    ),
)

handles = [
    plt.Line2D(
        [0],
        [0],
        marker="^",
        color="w",
        markerfacecolor=cp.TYPE_COLORS["waves"],
        markersize=10,
        linestyle="None",
        label=f"offshore MHL Waverider ({len(offshore_sites)})",
    ),
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=cp.TYPE_COLORS["calibration"],
        markersize=9,
        linestyle="None",
        label=f"nearshore Spotter ({len(nearshore_sites)})",
    ),
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor=cp.TYPE_COLORS["validation"],
        markersize=10,
        linestyle="None",
        label="spectral validation buoy",
    ),
]
ax.legend(handles=handles, loc="lower left", fontsize=9.5, framealpha=0.92)

x0, x1, y0, y1 = course_map_extent()
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)

save_fig(fig, "fig10_buoy_network")

Offshore MHL buoys      : 7
Nearshore Spotter buoys : 28
saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig10_buoy_network.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig10_buoy_network.png')

## Fig 11 — Reconstruction spectra network

Reconstructed **spectra bubbles** stamped directly onto the map at ~25
well-spaced sites along the 5 m isobath, together with the same WHACS
partition **forcing arrows** at the goals used in Fig 8 — one big figure
showing input (arrows) and output (spectra, but now spread across the whole
domain instead of at a single buoy) at the same moment.

Only a single day (24 hours, containing `moment`) is reconstructed for these
sites — the cost here is dominated by per-goal model loading, not by the
number of days, so restricting the time window is what keeps this affordable.
The site list itself is spaced to ~25 points precisely so the bubbles stay
legible instead of overlapping (reconstructing all ~1,400 5 m-isobath sites
at once was tried and is impractically slow for a notebook figure).

**Budget ~3–5 minutes the first time** this cell runs (one fresh
`mm.reconstruct(spectral=True, ...)` call); cached after that via
`tmp/reconstruct/`, like Fig 6/8/9's spectra.

In [13]:
apply_slide_style()
moment = resolve_forcing_moment()
day = moment.strftime("%Y-%m-%d")
print(f"Spectra network moment: {moment}  (reconstructing day {day})")


def pick_spaced_sites(
    lons: np.ndarray, lats: np.ndarray, min_spacing_deg: float
) -> np.ndarray:
    """Greedy latitude-ordered subsample keeping a minimum on-map spacing.

    Checked against *every* previously kept point (not just the last one),
    since NSW's isobath sites zigzag enough in longitude that a simple
    running-neighbour check lets clusters of near-duplicate points through.
    """
    order = np.argsort(lats)[::-1]  # north to south, matches G1..G26
    picked: list[int] = []
    pts: list[np.ndarray] = []
    for idx in order:
        pt = np.array([lons[idx], lats[idx]])
        if not pts or all(np.hypot(*(pt - p)) >= min_spacing_deg for p in pts):
            picked.append(idx)
            pts.append(pt)
    return np.array(picked)


active_goals = list(goals.keys())
template = mm.load_cases_goal(active_goals[0])
common_5m = [s for s in mm.common_model_sites(active_goals) if s.startswith("5m_")]
lon_5m = template.site_longitude.sel(sites=common_5m).values
lat_5m = template.site_latitude.sel(sites=common_5m).values

keep = pick_spaced_sites(lon_5m, lat_5m, min_spacing_deg=0.4)
network_site_ids = [common_5m[i] for i in keep]
print(
    f"Spectra network sites: {len(network_site_ids)} (of {len(common_5m)} on 5 m isobath)"
)

network_kw = dict(
    n=mda_n,
    offshore_name=offshore_name,
    goals=None,
    wl=wl,
    pca_variance=pca_variance,
)

mm.reconstruct(
    day, spectral=True, sites=network_site_ids, save_spectra=True, **network_kw
)
network_spec = mm.load_reconstruction_spectra(
    day, spectral=True, site_ids=network_site_ids, **network_kw
)
if network_spec is None:
    raise FileNotFoundError(
        f"Missing spectra cache for {day!r} — check mm.reconstruct(save_spectra=True) above."
    )
network_efth = network_spec["efth"].sel(time=moment, method="nearest")

fig, ax = plt.subplots(figsize=(18.0, 14.0), constrained_layout=True)
cp.plot_base_map(ax=ax, title=None, show_topobathy=True)
draw_goal_partition_arrows(ax, moment)

for site_id in network_site_ids:
    da = network_efth.sel(sites=site_id)
    if not np.isfinite(da.values).any() or float(da.max()) <= 0:
        continue
    lon = float(network_spec.site_longitude.sel(sites=site_id))
    lat = float(network_spec.site_latitude.sel(sites=site_id))
    img = render_spectrum_bubble(da, figsize=(1.9, 1.9))
    ax.plot(
        lon, lat, "o", color="white", markeredgecolor="black", markersize=4, zorder=7
    )
    ax.add_artist(
        AnnotationBbox(
            OffsetImage(np.asarray(img), zoom=0.24),
            (lon, lat),
            frameon=False,
            pad=0,
            zorder=6,
        )
    )

buoy_meta = obs.get_coordinates()[buoy_instrument]
ax.plot(
    buoy_meta["lon"],
    buoy_meta["lat"],
    marker="s",
    color=cp.TYPE_COLORS["validation"],
    markersize=14,
    markeredgecolor="white",
    markeredgewidth=0.9,
    zorder=9,
)
ax.annotate(
    "validation buoy",
    (buoy_meta["lon"], buoy_meta["lat"]),
    xytext=(8, 8),
    textcoords="offset points",
    fontsize=11,
    color=cp.TYPE_COLORS["validation"],
    fontweight="bold",
    zorder=10,
    bbox=dict(
        boxstyle="round,pad=0.2", fc="white", ec=cp.TYPE_COLORS["validation"], alpha=0.9
    ),
)

x0, x1, y0, y1 = course_map_extent()
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)
ax.set_aspect("equal", adjustable="box")
# plot_base_map(title=None) falls back to a default "Case map" title rather
# than omitting one — overwrite it here, same as every other map cell in
# this notebook that also passes title=None and sets its own title after.
ax.set_title(
    "INPUT — partition arrows at each offshore goal   |   "
    "OUTPUT — reconstructed spectra along the 5 m isobath",
    fontsize=13,
    pad=10,
    color="#1a1a2e",
)

legend_handles = partition_legend_handles() + [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=7,
        linestyle="None",
        label=f"reconstructed spectrum ({len(network_site_ids)} sites, 5 m isobath)",
    )
]
# "outside right lower" (not "upper") — a single full-height map axes plus
# its own colorbar leaves no spare row above the axes the way Fig 8's 3-panel
# spectra row does, so an upper-right legend collides with the suptitle here.
leg = fig.legend(
    handles=legend_handles,
    loc="outside right lower",
    fontsize=9.5,
    framealpha=0.96,
    edgecolor="#bbbbbb",
    title="WHACS partition forcing (input) &\nreconstructed spectra (output)",
    title_fontsize=10.5,
    ncol=1,
    borderpad=0.8,
)
leg.get_title().set_fontweight("bold")

fig.suptitle(
    f"RECONSTRUCTION SPECTRA NETWORK  |  {moment:%Y-%m-%d %H:%M} UTC",
    fontsize=17,
    fontweight="bold",
)

save_fig(fig, "fig11_reconstruction_spectra_network")

Spectra network moment: 2011-05-25 19:00:00  (reconstructing day 2011-05-25)
Spectra network sites: 25 (of 1384 on 5 m isobath)
saved → /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig11_reconstruction_spectra_network.png


PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig11_reconstruction_spectra_network.png')

## Export summary


In [14]:
sorted(EXPORT_DIR.glob("fig*.png"))

[PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig01_domain.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig02_offshore_partitions_ts.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig03_metamodel_lhs.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig04_whacs_hindcast_forcing.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig05_buoy_context_map.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig06_hero_spectrum_storm_peak.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig07_pipeline_schematic.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figures/fig08_input_output_partitions.png'),
 PosixPath('/home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/gui_figur